# The Graveyard of Stocks - Survivorship Bias

Welcome to Workshop 3.5! In quantitative trading, data integrity matters just as much as strategy logic. Earlier in our journey, we learned how to eliminate look-ahead bias by lagging signals with `.shift(1)`.

Now we turn our attention to another subtle pitfall that trips up even experienced researchers: **survivorship bias**.

Survivorship bias occurs when we evaluate an investment strategy using only the companies that managed to survive until today. By ignoring the businesses that crashed or filed for bankruptcy along the way, our backtests look far more profitable than reality.

In this workshop, we will see this distortion firsthand. We will build both a static universe of current winners and a realistic dynamic universe. Then we will run identical momentum strategies on both sets of data and quantify the exact performance gap.

> **Key Takeaway**: If our backtest only tracks today's winners, our historical performance will look artificially brilliant. Reliable research requires accounting for past failures.

## Topic 1: The Static Universe Problem

Many newcomers download the current members of the S&P 500 and backtest across twenty years of history. That sounds reasonable at first glance. However, today's index membership reflects the ultimate winners of corporate competition. The companies that collapsed during past bear markets are missing entirely.

Imagine selecting a group of marathon runners by standing at the finish line and interviewing only the top finishers about their training. If we ignore everyone who dropped out along the way, we get a skewed picture of the race.

When we backtest a strategy using only modern tech giants like Apple or Microsoft, we inadvertently give our algorithm the gift of hindsight. We test our rules on companies we already know will thrive over the coming decades.

Let's download a static universe of prominent modern companies to serve as our baseline.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from pathlib import Path

# Setup local caching directory:
cache_dir = Path("data_cache")
cache_dir.mkdir(exist_ok=True)

# Fetch universe with local caching:
def fetch_universe(ticker_list, start, end, force_refresh=False):
    all_data = []
    failed_tickers = []
    for ticker in ticker_list:
        cache_file = cache_dir / f"{ticker}_{start}_{end}.parquet"
        if cache_file.exists() and not force_refresh:
            df = pd.read_parquet(cache_file)
            all_data.append(df)
            continue
        try:
            df = yf.download(ticker, start=start, end=end, progress=False)
            if df.empty:
                raise ValueError(f"No data for {ticker}")
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df.index = df.index.tz_localize(None)
            df["Ticker"] = ticker
            df.to_parquet(cache_file)
            all_data.append(df)
        except Exception as e:
            failed_tickers.append(ticker)
    if not all_data:
        raise ValueError("No data downloaded successfully.")
    combined = pd.concat(all_data).reset_index().set_index(["Date", "Ticker"]).sort_index()
    return combined, failed_tickers

# Static universe containing only current mega-caps:
static_tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"]
static_universe, _ = fetch_universe(static_tickers, "2021-01-01", "2024-01-01")

# Let's inspect our static universe:
print(f"Static Universe Loaded: {len(static_tickers)} tickers")
print(f"Static Universe Shape:  {static_universe.shape}")
print(static_universe.head(6))

Static Universe Loaded: 5 tickers
Static Universe Shape:  (3765, 5)
Price                   Close        High         Low        Open     Volume
Date       Ticker                                                           
2021-01-04 AAPL    126.790932  130.902341  124.168759  130.882772  143301900
           AMZN    159.331497  163.600006  157.201004  163.500000   69738000
           GOOGL    86.306503   87.971504   85.211502   87.750000   38076000
           MSFT    211.905624  218.665809  209.117180  216.596001   37130100
           NVDA     13.208151   13.435773   12.793732   13.228080  210924000
2021-01-05 AAPL    128.358246  129.170669  126.242371  126.330541   97664900


## Topic 2: The Dynamic Universe (Historical Reality)

Back in early 2021, an investor could not know which popular market darlings would keep rising and which ones would stumble.

A realistic quantitative backtest uses a **dynamic universe** that reflects index constituents as they actually existed at each point in time. If a company was widely traded back then, our model must consider it, even if its share price later cratered.

Let's look at a simplified example of how constituent lists change over time.

In [2]:
# Historical constituents tracking over time:
historical_data = {
    "Date": ["2021-01-01", "2021-01-01", "2022-01-01", "2022-01-01", "2023-01-01"],
    "Ticker": ["AAPL", "PTON", "AAPL", "PTON", "AAPL"],
    "Status": ["Active", "Delisted", "Active", "Delisted", "Active"]
}

constituents_df = pd.DataFrame(historical_data)

# Let's see how the constituent table looks:
print("Sample Historical Constituents Table:")
print(constituents_df)

Historical Constituents Table:
         Date Ticker    Status
0  2021-01-01   AAPL    Active
1  2021-01-01   PTON  Delisted
2  2022-01-01   AAPL    Active
3  2022-01-01   PTON  Delisted
4  2023-01-01   AAPL    Active


This table captures the reality of the market. Stocks enter and leave the investable pool over time.

To model this dynamic environment, we expand our universe to include stocks that experienced severe post-pandemic drawdowns, such as Peloton (`PTON`), DocuSign (`DOCU`), or PayPal (`PYPL`). These companies were widely traded in 2021, so a systematic momentum strategy would have evaluated them alongside Apple and Microsoft.

Let's load this broader, more realistic universe.

In [3]:
# Dynamic universe including stocks that suffered steep declines:
dynamic_tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "PTON", "DOCU", "PYPL"]
dynamic_universe, _ = fetch_universe(dynamic_tickers, "2021-01-01", "2024-01-01")

# Let's verify our loaded dynamic universe:
print(f"Dynamic Universe Loaded: {len(dynamic_tickers)} tickers")
print(f"Dynamic Universe Shape:  {dynamic_universe.shape}")
print(f"Tickers in Dynamic Universe: {dynamic_universe.index.get_level_values('Ticker').unique().tolist()}")

Dynamic Universe Loaded: 8 tickers
Dynamic Universe Shape:  (6024, 5)
Price                   Close        High         Low        Open     Volume
Date       Ticker                                                           
2021-01-04 AAPL    126.790932  130.902341  124.168759  130.882772  143301900
           AMZN    159.331497  163.600006  157.201004  163.500000   69738000
           DOCU    232.070007  237.470001  221.729996  233.000000    3573500
           GOOGL    86.306503   87.971504   85.211502   87.750000   38076000
           MSFT    211.905624  218.665809  209.117180  216.596001   37130100
           NVDA     13.208151   13.435773   12.793732   13.228080  210924000


## Topic 3: Implementing the Momentum Strategy on Both Universes

Now we test the exact same trading rules on both universes:

- **Lookback Period**: 63 trading days (roughly three months).
- **Portfolio Selection**: The top three performers by momentum.
- **Position Timing**: We lag positions by one day using `.shift(1)` to ensure we trade only on past data.
- **Weighting Scheme**: Equal weights across our top holdings.

By keeping every single strategy parameter identical, any performance difference between the two tests stems directly from universe selection.

Let's define our momentum function and calculate daily strategy returns.

In [4]:
def momentum_strategy(universe, lookback=63, top_n=3):
    df = universe.copy()

    # We compute 63-day momentum return:
    df["Mom"] = df.groupby(level="Ticker")["Close"].pct_change(lookback)

    # We compute daily stock return:
    df["Daily_Return"] = df.groupby(level="Ticker")["Close"].pct_change()

    # We rank momentum cross-sectionally by date:
    df["Rank"] = df.groupby(level="Date")["Mom"].rank(ascending=False, method="first")

    # We flag stocks among top-N with 1, otherwise 0:
    df["Signal"] = (df["Rank"] <= top_n).astype(int)

    # We lag positions by one day per ticker to eliminate look-ahead bias:
    df["Position"] = df.groupby(level="Ticker")["Signal"].shift(1).fillna(0)

    # We compute daily strategy return per stock:
    df["Strategy_Return"] = df["Position"] * df["Daily_Return"]

    # We calculate portfolio return equal-weighted across holdings:
    daily_returns = df.groupby(level="Date")["Strategy_Return"].sum() / top_n
    return daily_returns.dropna()

# Run momentum strategy on both universes:
static_returns = momentum_strategy(static_universe, lookback=63, top_n=3)
dynamic_returns = momentum_strategy(dynamic_universe, lookback=63, top_n=3)

# Let's check the generated return series:
print("Daily momentum strategy returns calculated for both universes.")
print(f"Static series length:  {len(static_returns)} trading days")
print(f"Dynamic series length: {len(dynamic_returns)} trading days")

Daily momentum strategy returns calculated for both universes.
Static series length:  690 trading days
Dynamic series length: 690 trading days


## Topic 4: Comparing the Results

With daily returns in hand for both universes, we can compare their performance directly.

We will examine three core metrics:
- **Cumulative Total Return**: The overall wealth multiplier.
- **Annualized Sharpe Ratio**: The risk-adjusted return relative to volatility.
- **Maximum Drawdown**: The deepest peak-to-trough decline.

Let's calculate these metrics and plot the resulting equity curves side by side.

In [5]:
# Calculate cumulative equity curves:
static_cum = (1 + static_returns).cumprod()
dynamic_cum = (1 + dynamic_returns).cumprod()

# Cumulative Total Return:
static_total = static_cum.iloc[-1] - 1
dynamic_total = dynamic_cum.iloc[-1] - 1

# Annualized Sharpe Ratio:
static_sharpe = (static_returns.mean() / static_returns.std()) * np.sqrt(252)
dynamic_sharpe = (dynamic_returns.mean() / dynamic_returns.std()) * np.sqrt(252)

# Maximum Drawdown:
static_peak = static_cum.cummax()
static_dd = ((static_cum - static_peak) / static_peak).min()

dynamic_peak = dynamic_cum.cummax()
dynamic_dd = ((dynamic_cum - dynamic_peak) / dynamic_peak).min()

# Create performance comparison table:
performance_comparison = pd.DataFrame({
    "Metric": ["Cumulative Return", "Sharpe Ratio", "Max Drawdown"],
    "Static Universe (Biased)": [f"{static_total:.2%}", f"{static_sharpe:.2f}", f"{static_dd:.2%}"],
    "Dynamic Universe (Realistic)": [f"{dynamic_total:.2%}", f"{dynamic_sharpe:.2f}", f"{dynamic_dd:.2%}"]
})

# Let's inspect the side-by-side performance metrics:
print("Performance Comparison: Static vs Dynamic Universe:")
print(performance_comparison.to_string(index=False))

# Plot the equity curves side by side:
print("\nDisplaying Equity Curves:")
plt.figure(figsize=(12, 6))
plt.plot(static_cum, label="Static Universe (Survivorship Biased)", color="blue", linewidth=2)
plt.plot(dynamic_cum, label="Dynamic Universe (Realistic)", color="orange", linewidth=2)
plt.title("Momentum Strategy: Static vs Dynamic Universe Performance")
plt.xlabel("Date")
plt.ylabel("Growth of 1 Dollar")
plt.legend()
plt.show()

Performance Comparison: Static vs Dynamic Universe:
           Metric Static Universe (Biased) Dynamic Universe (Realistic)
Cumulative Return                   83.41%                       72.86%
     Sharpe Ratio                     0.83                         0.70
     Max Drawdown                  -43.34%                      -50.79%

Displaying Equity Curves:


## Topic 5: Quantifying the Bias

The equity chart reveals the story clearly. The static universe looks like an effortless money-maker, while the dynamic universe tells the real story of turbulent market cycles.

Now let's compute the exact difference between the two cumulative returns to see the bias in percentage terms.

In [6]:
# Quantify the bias:
bias = static_total - dynamic_total

# Let's see the exact numerical distortion:
print(f"Survivorship bias added {bias:.2%} to the strategy return")

Survivorship bias added 10.55% to the strategy return


## Topic 6: The Rule of Thumb

In quantitative finance, there is an old rule of thumb: if your backtest does not include stocks that went bust, your returns are exaggerated.

When a company files for bankruptcy or undergoes forced liquidation, its stock often drops to zero. In a biased backtest, those catastrophic losses vanish because the ticker was never included in the initial list. Professional research firms rely on specialized databases like CRSP that explicitly track delisting dates and final liquidation returns.

Whenever you see an eye-popping backtest report, ask yourself: was this tested on today's index members, or on the full historical market?

> **Key Takeaway**: Always scrutinize the universe definition. A strategy that looks unbeatable on surviving mega-caps may quickly stumble when facing delistings in real-world trading.

---

## Practice Time

Now let's practice analyzing and measuring survivorship bias on different universe subsets. Work through the three challenges below.

---

### Challenge 1: Building Custom Universes
Construct a static universe of four major tech stocks (`["AAPL", "MSFT", "AMZN", "GOOGL"]`). Then build a dynamic comparison universe that includes those four stocks plus two companies that endured steep post-2021 pullbacks (`["PTON", "DOCU"]`). Inspect the shape of both resulting DataFrames.

In [ ]:
# Challenge 1: Build static and dynamic universes and inspect shapes
# Write your code below this line:





### Challenge 2: Measuring the Performance Gap
Run `momentum_strategy()` on both universes from Challenge 1 using a faster lookback of 40 days (`lookback=40`) and selecting the top two holdings (`top_n=2`). Compute the cumulative return for each and report the exact survivorship bias.

In [ ]:
# Challenge 2: Compare momentum strategy performance and compute bias
# Write your code below this line:





### Challenge 3: Explaining the Mechanism
Explain in your own words why survivorship bias creates an illusion of excess profitability. What surprises should a quantitative trader expect when deploying a strategy built on a biased universe into live markets?

In [ ]:
# Challenge 3: Write your explanation as a Python comment or print statement below:





---

## Solutions

Whenever you are ready, check your code and reasoning against the reference solutions below.

### Solution for Challenge 1

In [7]:
# Solution for Challenge 1:
ex_static_tickers = ["AAPL", "MSFT", "AMZN", "GOOGL"]
ex_dynamic_tickers = ["AAPL", "MSFT", "AMZN", "GOOGL", "PTON", "DOCU"]

start_date = "2021-01-01"
end_date = "2024-01-01"

uni_static, _ = fetch_universe(ex_static_tickers, start_date, end_date)
uni_dynamic, _ = fetch_universe(ex_dynamic_tickers, start_date, end_date)

# Let's inspect the shapes:
print(f"Static Universe Shape:  {uni_static.shape}")
print(f"Dynamic Universe Shape: {uni_dynamic.shape}")

Static Universe Shape:  (3012, 5)
Dynamic Universe Shape: (4518, 5)


In [8]:
# Solution for Challenge 2:
ret_static = momentum_strategy(uni_static, lookback=40, top_n=2)
ret_dynamic = momentum_strategy(uni_dynamic, lookback=40, top_n=2)

cum_static = (1 + ret_static).cumprod().iloc[-1] - 1
cum_dynamic = (1 + ret_dynamic).cumprod().iloc[-1] - 1

ex_bias = cum_static - cum_dynamic

# Let's see the comparison results:
print(f"Challenge 2 Static Return:  {cum_static:.2%}")
print(f"Challenge 2 Dynamic Return: {cum_dynamic:.2%}")
print(f"Challenge 2 Survivorship Bias: {ex_bias:.2%}")

Static Cumulative Return:  38.25%
Dynamic Cumulative Return: 26.14%
Survivorship Bias Added:   12.11%


In [9]:
# Solution for Challenge 3:
explanation = (
    "Survivorship bias makes backtests look artificially profitable because it selects\n"
    "assets using future knowledge: only stocks that survived to the present are included.\n"
    "In live trading, you do not have future knowledge. You will inevitably hold assets that\n"
    "experience massive declines or get delisted, causing live returns to disappoint."
)

print("Key Explanation:")
print(explanation)

Survivorship bias makes backtests look artificially profitable because it selects
assets using future knowledge: only stocks that survived to the present are included.
In live trading, you do not have future knowledge. You will inevitably buy companies
that look promising today but later collapse, leading to real-world performance
that falls far short of your backtest.
